# Ensemble Retriever — 하이브리드 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Sparse(BM25)**와 **Dense(벡터)** 검색 방식의 차이와 각각의 강점 이해하기
- `EnsembleRetriever`로 두 방식을 결합하는 방법 익히기
- `weights` 파라미터로 가중치를 조정해 검색 성능 튜닝하기

## 사전 지식

- BM25: 키워드 빈도를 활용한 전통적 검색 방식
- Dense Retrieval: 임베딩 벡터 간 유사도 기반 검색

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> V[FAISS<br/>의미 검색]:::process
    B -- 가중치 0.5 --> E[EnsembleRetriever<br/>RRF 알고리즘]:::process
    V -- 가중치 0.5 --> E
    E --> R[최적 검색 결과]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 아이디어

| 특성 | Sparse / BM25 | Dense / Vector |
|------|:---:|:---:|
| 기반 | 키워드 매칭 | 의미 유사도 |
| 강점 | 정확한 용어 매칭 | 문맥·동의어 이해 |
| 약점 | 동의어 못 찾음 | 키워드 민감도 낮음 |

두 방식을 결합하면 각각의 약점을 보완해요. **RRF(Reciprocal Rank Fusion)** 알고리즘으로 결과를 통합합니다:

```
score = Σ(weight_i / (k + rank_i))
```

## 환경 설정


In [ ]:
from dotenv import load_dotenv

load_dotenv()


## 1. 문서 준비 및 개별 Retriever 생성


In [ ]:
# ---------------------------------------------------
# 하이브리드 검색을 위한 문서 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 500자 청크로 분할하세요
# 힌트: TextLoader → load() → RecursiveCharacterTextSplitter → split_documents()
# 예상 결과: 분할된 청크 수 출력
# ============================================================

# 문서 로드 및 분할
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

print(f"✅ 문서 준비 완료: {len(split_docs)}개 청크")

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1. BM25 Retriever (Sparse - 키워드 기반)
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

# 2. FAISS Retriever (Dense - 의미 기반)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ BM25 Retriever 생성 완료 (키워드 기반)")
print("✅ FAISS Retriever 생성 완료 (의미 기반)")


In [ ]:
# ---------------------------------------------------
# 개별 Retriever로 동일 쿼리를 검색하여 차이점 확인
# ---------------------------------------------------

# ============================================================
# TODO: bm25_retriever와 faiss_retriever에 동일한 쿼리를 각각 실행하세요
# 힌트: retriever.invoke(query) 후 결과 출력
# 예상 결과: BM25는 키워드 매칭, FAISS는 의미 유사도 기반 문서 반환
# ============================================================

# 개별 retriever 성능 비교
query = "딥러닝 모델에서 가중치 학습은 어떻게 이루어지나요?"



print(f"📝 쿼리: {query}\n")
print("="*80)
print("[BM25 결과 - 키워드 기반]")
print("="*80)
for i, doc in enumerate(bm25_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("="*80)
print("[FAISS 결과 - 의미 기반]")
print("="*80)
for i, doc in enumerate(faiss_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n💡 관찰:")
print("  - BM25: '학습', '가중치' 등 키워드가 명확한 문서 우선")
print("  - FAISS: 전체 문맥과 의미가 유사한 문서 우선")

## 2. EnsembleRetriever 생성 및 사용

두 retriever를 결합하여 각각의 강점을 활용합니다.


In [ ]:
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever 생성 (가중치 5:5)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

# Ensemble 검색
ensemble_docs = ensemble_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[Ensemble 결과 - 하이브리드 검색]")
print("="*80)
for i, doc in enumerate(ensemble_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n💡 장점:")
print("  - BM25와 FAISS의 결과를 모두 반영")
print("  - RRF 알고리즘으로 최적 순위 결정")
print("  - 더 다양하고 관련성 높은 결과")

In [ ]:
# ---------------------------------------------------
# EnsembleRetriever의 가중치를 변경하여 결과 차이 실험
# ---------------------------------------------------

# ============================================================
# TODO: BM25 우선(7:3)과 FAISS 우선(3:7) 두 가지 가중치로 같은 쿼리를 검색하세요
# 힌트: EnsembleRetriever(retrievers=[...], weights=[w1, w2])
# 예상 결과: 가중치에 따라 상위 문서 순서가 달라짐
# ============================================================

# 가중치 실험
test_query = "Transformer 아키텍처의 핵심 요소는?"

# 케이스 1: BM25 우선 (7:3)
# 전문 용어가 많거나 정확한 키워드 매칭이 중요할 때


# 케이스 2: FAISS 우선 (3:7)
# 자연어 질문이 많거나 동의어를 고려해야 할 때


docs_bm25_heavy = ensemble_bm25_heavy.invoke(test_query)
docs_faiss_heavy = ensemble_faiss_heavy.invoke(test_query)

print(f"📝 쿼리: {test_query}\n")
print("="*80)
print("[BM25 우선 (7:3) - 키워드 중시]")
print("="*80)
for i, doc in enumerate(docs_bm25_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

print("="*80)
print("[FAISS 우선 (3:7) - 의미 중시]")
print("="*80)
for i, doc in enumerate(docs_faiss_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

print("\n💡 가중치 선택 가이드:")
print("  - 전문 용어 많음 → BM25 가중치 높임 (0.6-0.7)")
print("  - 자연어 질문 → FAISS 가중치 높임 (0.6-0.7)")
print("  - 일반적 사용 → 균등 (0.5-0.5)")

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| 가중치 설정 | 상황 | 예시 |
|-----------|------|------|
| `[0.7, 0.3]` | 전문 용어·고유명사 중심 | 의학, 법률 문서 |
| `[0.5, 0.5]` | 일반적 사용 (기본 권장) | 뉴스, 블로그 |
| `[0.3, 0.7]` | 자연어 질문·동의어 많음 | 고객 상담, Q&A |

### 다음 노트북 예고

**04-LongContextReorder**: LLM이 긴 컨텍스트의 중간 부분을 잘 활용하지 못하는 "Lost-in-the-Middle" 문제와 해결법을 배워요.